# Mantis Vision — Train the multi-head health model in Google Colab

Run these cells top to bottom. Before starting: **Runtime -> Change runtime type -> GPU**.

This notebook trains the **multi-head** health classifier: one EfficientNet-B0 backbone with three heads —
**category** (Healthy/Moderate/Low), **condition** (None/Dried/Decayed/Diseased, shown only when category != Healthy),
and a **health score regressed to 0–10**. It also adds a confidence-calibration step (is the reported confidence
trustworthy, or just a raw number?) and a bullet-point explanation preview.

Checkpoints from the older single-head notebook (`MantisVision_Training.ipynb`) are **not compatible** with this
model — that notebook is superseded by this one.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/Arran0/MantisVision.git
%cd MantisVision/ml

## 2. Install dependencies

Colab already has torch/torchvision preinstalled with GPU support, so this just adds the extra packages the pipeline needs.

In [ ]:
!pip install -q grad-cam fastapi python-multipart

## 3. Upload your dataset

On your own computer, organize your photos into `train/validation/test/<Class>/` folders (see
`docs/DATASET_LABELING_GUIDE.md`) — each split (`train`, `validation`, `test`) contains one subfolder per **raw**
class (`Healthy`, `Moderate`, `Low`, `Decay`, `Dried`, `Disease`) full of images. Keep the class subfolders intact;
don't flatten them out. You do **not** need to nest them under a species folder yourself — the cell below extracts
them straight into the species-scoped path the pipeline expects (`config.dataset_dir`, currently
`kappaphycus_alvarezii/`).

Then select the **`train`, `validation`, and `test` folders themselves** (all three, as siblings) and zip them
together into one `.zip`. Do **not** zip only the loose images, and do **not** zip a parent folder that wraps
around `train`/`validation`/`test` — when someone unzips your file, `train/`, `validation/`, and `test/` should
appear immediately, not nested inside another folder. (If you do end up with an extra wrapper level anyway, or a
class folder is called `val`/`valid`/`validate` instead of `validation`, or the zip carries macOS
`__MACOSX`/`.DS_Store` clutter, the cell below auto-detects and fixes all of that.)

Run the cell below, then use the file picker to select that `.zip`.

In [ ]:
import pathlib
import shutil
import zipfile

from google.colab import files

import sys
sys.path.insert(0, ".")
from config import config

dataset_dir = config.dataset_dir  # species-scoped, e.g. dataset/kappaphycus_alvarezii/
dataset_dir.mkdir(parents=True, exist_ok=True)

print(f"Extracting into {dataset_dir} ...")
print("Select the zip file containing your train/validation/test folders...")
uploaded = files.upload()

for name in uploaded:
    if name.lower().endswith(".zip"):
        print(f"Extracting {name} ...")
        with zipfile.ZipFile(name) as zf:
            zf.extractall(dataset_dir)
    else:
        print(f"Skipping non-zip file: {name} (expected a single .zip upload)")

# Clean up macOS zip artifacts: the __MACOSX/ sidecar folder plus .DS_Store
# and AppleDouble "._*" files that macOS's Archive Utility sprinkles into
# zips made on a Mac. None of these are real images, so they'd otherwise
# just sit alongside the class folders as clutter.
shutil.rmtree(dataset_dir / "__MACOSX", ignore_errors=True)
for junk in list(dataset_dir.rglob(".DS_Store")) + list(dataset_dir.rglob("._*")):
    if junk.is_file():
        junk.unlink()

# Auto-fix: find wherever train/validation/test actually ended up, however
# deep, and however "validation" was spelled (val/valid/validate are all
# accepted), then move+rename them into dataset_dir/{train,validation,test}.
# Handles zips that wrap everything in a parent folder (e.g. the zip's name)
# as well as folders named e.g. "validate" instead of "validation". The repo
# already ships empty dataset/<species_slug>/{train,validation,test}/<Class>/.gitkeep
# placeholders, so we can't just check "do these three folders exist" — we
# specifically look for the shallowest match that actually contains images.
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp"}
split_aliases = {
    "train": {"train", "training"},
    "validation": {"validation", "val", "valid", "validate"},
    "test": {"test", "testing"},
}


def canonical_split(dirname):
    lowered = dirname.lower()
    for canonical, aliases in split_aliases.items():
        if lowered in aliases:
            return canonical
    return None


def has_images(path):
    return any(f.is_file() and f.suffix.lower() in IMAGE_EXTS for f in path.rglob("*"))


candidates = sorted(
    [dataset_dir] + [p for p in dataset_dir.rglob("*") if p.is_dir()],
    key=lambda p: len(p.relative_to(dataset_dir).parts),
)

holder, matches = None, {}
for candidate in candidates:
    found = {}
    for child in candidate.iterdir():
        if not child.is_dir():
            continue
        canonical = canonical_split(child.name)
        if canonical and canonical not in found:
            found[canonical] = child
    if {"train", "validation", "test"}.issubset(found) and any(has_images(d) for d in found.values()):
        holder, matches = candidate, found
        break

if holder is not None:
    if holder != dataset_dir:
        print(f"Found the splits inside '{holder.relative_to(dataset_dir)}/', moving them up...")
    for canonical, child in matches.items():
        dest = dataset_dir / canonical
        if child.resolve() == dest.resolve():
            continue
        if child.name != canonical:
            print(f"Renaming '{child.name}/' -> '{canonical}/'")
        if dest.exists():
            shutil.copytree(child, dest, dirs_exist_ok=True)
            shutil.rmtree(child)
        else:
            shutil.move(str(child), str(dest))
    if holder != dataset_dir:
        shutil.rmtree(holder, ignore_errors=True)
else:
    print("Warning: could not find any images under train/validation/test folders in the zip.")


def print_tree(path, prefix="", depth=0, max_depth=3):
    for entry in sorted(path.iterdir()):
        print(f"{prefix}{entry.name}{'/' if entry.is_dir() else ''}")
        if entry.is_dir() and depth < max_depth:
            print_tree(entry, prefix + "  ", depth + 1, max_depth)


print(f"\nContents of {dataset_dir.resolve()}:\n")
print_tree(dataset_dir)

## 4. Validate the dataset

Checks every raw class folder exists, every image opens correctly, and flags underrepresented classes.

In [ ]:
!python -m src.data.validate_dataset

## 5. Preview the category/condition/score mapping

Before spending GPU time, sanity-check what the model is about to be trained to predict. Each raw folder
(`Healthy`/`Moderate`/`Low`/`Decay`/`Dried`/`Disease`) is remapped into (category, condition, health-score anchor)
via `config.CLASS_TARGET_MAP` — see `docs/DATASET_LABELING_GUIDE.md` for the justification. **The anchors are a
documented heuristic, not measured biological ground truth** — no expert-scored 0–10 dataset exists yet.

In [ ]:
from config import CLASS_TARGET_MAP, config

print(f"{'Raw folder':<12}{'Category':<12}{'Condition':<12}{'Score anchor':>13}")
for raw_name in config.class_names:
    t = CLASS_TARGET_MAP[raw_name]
    print(f"{raw_name:<12}{t.category:<12}{(t.condition or '-'):<12}{t.score_anchor:>13.1f}")

## 6. Train

Two-phase schedule (frozen backbone, then fine-tune) with early stopping, training all three heads jointly
(category cross-entropy + masked condition cross-entropy + score regression). Saves the best checkpoint to
`ml/checkpoints/best_model.pt`. This can take a while depending on dataset size — Colab's free GPU tier helps a
lot here.

In [ ]:
!python -m src.train

## 7. Evaluate

Per-head metrics: category accuracy/precision/recall/F1/confusion matrix/ROC-AUC, condition metrics (non-Healthy
samples only), and health-score MAE/RMSE/R² against the anchor targets — plus a raw (uncalibrated) confidence
check (Expected Calibration Error).

In [ ]:
!python -m src.evaluate

from IPython.display import Image, display
display(Image(filename="reports/confusion_matrix_category.png"))
display(Image(filename="reports/confusion_matrix_condition.png"))
display(Image(filename="reports/score_scatter.png"))
display(Image(filename="reports/score_by_raw_class.png"))

## 8. Check whether "confidence" is trustworthy

This is the concrete answer to *"is the confidence number real, or is it just making things up?"* The raw
`confidence` the model reports **is** a genuine softmax probability — it's not fabricated — but a model trained
with plain cross-entropy is often overconfident: its stated confidence doesn't necessarily match its real accuracy.

This step fits a **temperature-scaling** calibration on the validation split, then reports **Expected Calibration
Error (ECE)** on the held-out test split before and after, with reliability diagrams for both. If ECE drops after
calibration, the raw confidence was measurably miscalibrated and the fix helped; `checkpoints/calibration.json` is
what the inference API picks up to also return a `confidence_calibrated` field. Re-run this after every retrain —
it's not a one-time check.

In [ ]:
!python -m src.calibrate

from IPython.display import Image, display
import json

print(json.dumps(json.load(open("checkpoints/calibration.json")), indent=2))
display(Image(filename="reports/reliability_diagram_before.png"))
display(Image(filename="reports/reliability_diagram_after.png"))

## 9. Download the trained checkpoint

Colab runtimes are ephemeral — grab these now so you don't lose them when the session disconnects. You'll need
both files to run the inference API later (`calibration.json` is optional — without it, the API just returns
`confidence_calibrated: null`).

In [ ]:
from google.colab import files

files.download("checkpoints/best_model.pt")
files.download("checkpoints/calibration.json")

## 10. Try a full prediction on one photo

Upload a single test photo and see the complete output the API would return: species, category, condition,
health score (0–10), raw + calibrated confidence, preset explanation bullets, recommendation, and the Grad-CAM
heatmap (highlighting which pixels drove the *category* prediction).

In [ ]:
from google.colab import files

uploaded = files.upload()  # pick one photo
photo_path = list(uploaded.keys())[0]

import sys
sys.path.insert(0, ".")
from src.inference.predictor import Predictor

predictor = Predictor()
with open(photo_path, "rb") as f:
    result = predictor.predict(f.read())

print(f"Species:              {result.species}")
print(f"Category:             {result.category}")
print(f"Condition:            {result.condition}")
print(f"Health score (0-10):  {result.health_score:.1f}")
print(f"Confidence (raw):     {result.confidence:.3f}")
print(f"Confidence (calibrated): {result.confidence_calibrated}")
print("Explanation:")
for bullet in result.explanation_bullets:
    print(f"  - {bullet}")
print(f"Recommendation:       {result.recommendation}")

In [ ]:
!python -m src.gradcam "{photo_path}"

from IPython.display import Image, display
import pathlib
display(Image(filename=str(pathlib.Path(photo_path).with_suffix('.gradcam.png'))))